## Spark Beispiel mit RDDs

Für erste Experimente mit Apache Spark eignet sich Google Colab gut. Colab ermöglicht es, Python-Notebooks direkt im Browser auszuführen und benötigte Bibliotheken wie PySpark ohne lokale Installation zu verwenden.

Dadurch können die folgenden Beispiele zu RDDs und Transformationen in Spark unkompliziert ausprobiert und nachvollzogen werden.

In [2]:
from pyspark.sql import SparkSession

# Spark-Session starten
spark = SparkSession.builder \
    .appName("ReviewsExampleRDD") \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext

### RDD erstellen
Der Einfachheit halber wird hier ein RDD aus einer Liste erstellt.

In [3]:
reviews = [
(5,"Der Laptop ist schnell und leise"),
(4,"Gute Qualität"),
(1,"Gerät war defekt"),
(5,"Schnell geliefert"),
(3,"Preis etwas hoch")
]

reviews_rdd = sc.parallelize(reviews)

## Transformationen

Nach dem Einlesen der Eingabedaten werden mehrere Transformationen auf das RDD angewendet:

Zunächst werden ungültige Zeilen entfernt, die beim Parsen kein gültiges Tupel erzeugt haben. Anschließend werden nur positive Bewertungen mit mindestens vier Sternen ausgewählt. Aus diesen Bewertungen wird dann ausschließlich der Bewertungstext extrahiert.

Im nächsten Schritt werden die Texte in einzelne Wörter zerlegt (`flatMap`). Dadurch entsteht ein RDD, das jedes Wort der Bewertungen als eigenes Element enthält. Abschließend wird dieses RDD gefiltert, sodass nur noch das Wort **„schnell“** übrig bleibt.

Diese Pipeline demonstriert typische Spark-Transformationen (`filter`, `map`, `flatMap`), die nacheinander auf einem RDD angewendet werden.

In [9]:
# Ungültige Zeilen entfernen
parsed_rdd = reviews_rdd.filter(lambda x: x is not None)

# Nur positive Bewertungen
positive_rdd = parsed_rdd.filter(lambda x: x[0] >= 4)

# Nur den Bewertungstext behalten
texts_rdd = positive_rdd.map(lambda x: x[1])

# Wörter extrahieren
words_rdd = texts_rdd.flatMap(lambda text: text.lower().split(" "))

# Nur das Wort "schnell" herausfiltern
schnell_rdd = words_rdd.filter(lambda word: word == "schnell")

## Word Count mit `reduceByKey`

In diesem Schritt wird gezählt, wie oft jedes Wort vorkommt.

Zunächst wandelt `map` jedes Wort in ein Schlüssel-Wert-Paar der Form `(Wort, 1)` um.  
Anschließend fasst `reduceByKey` alle Paare mit dem gleichen Schlüssel zusammen und addiert ihre Werte. Dadurch entsteht für jedes Wort die Gesamtanzahl seiner Vorkommen.

In [10]:
word_counts_rdd = words_rdd.map(lambda w: (w, 1)) \
                           .reduceByKey(lambda a, b: a + b)

### Aktion `collect()`

Die Aktion `collect()` startet die Ausführung der zuvor definierten Transformationen.  
Dabei werden alle Ergebnisse der einzelnen Partitionen vom Cluster zum Driver-Prozess übertragen und als Liste im Programm zurückgegeben.

In [14]:
result = word_counts_rdd.collect()

### Alternativen zu `collect()`

Bei großen Datensätzen kann `collect()` sehr viel Speicher im Driver benötigen.  
Stattdessen können andere Aktionen verwendet werden:

- `take(n)`: Gibt die ersten *n* Elemente zurück.
- `saveAsTextFile(path)`: Speichert die Ergebnisse verteilt im Dateisystem.
- `takeSample()`: Liefert eine zufällige Stichprobe der Daten.

In [11]:
# Nur die ersten 10 Ergebnisse holen
word_counts_rdd.take(10)

# Beispiel: Ergebnisse verteilt speichern
# word_counts_rdd.saveAsTextFile("hdfs://daten/output/wordcounts")

# Beispiel: Zufallsstichprobe ziehen
# word_counts_rdd.takeSample(False, 10)

[('der', 1),
 ('laptop', 1),
 ('ist', 1),
 ('und', 1),
 ('leise', 1),
 ('gute', 1),
 ('geliefert', 1),
 ('schnell', 2),
 ('qualität', 1)]

### Partitionen anzeigen

Mit `getNumPartitions()` wird angezeigt, in wie viele Partitionen ein RDD aufgeteilt ist. Diese Partitionen bestimmen, wie die Daten im Cluster verteilt und parallel verarbeitet werden können.

In [12]:
words_rdd.getNumPartitions()

2

### Cache nutzen

Ein häufig genutztes Spark-Konzept ist das Zwischenspeichern von RDDs im Arbeitsspeicher. Durch `cache()` wird das RDD im Speicher der Executor gehalten. Beim ersten Aufruf von `count()` werden die Daten berechnet und im Cache abgelegt. Beim zweiten Aufruf kann Spark die Daten direkt aus dem Cache lesen, wodurch die Berechnung schneller erfolgt.


In [13]:
words_rdd.cache()
words_rdd.count()
words_rdd.count()

10

### RDD Lineage anzeigen

Mit `toDebugString()` kann die interne Struktur eines RDD angezeigt werden. Die Ausgabe zeigt die sogenannte Lineage, also die Abfolge von Transformationen, aus denen das aktuelle RDD entstanden ist.



In [15]:
print(word_counts_rdd.toDebugString().decode("utf-8"))

(2) PythonRDD[10] at RDD at PythonRDD.scala:56 []
 |  MapPartitionsRDD[4] at mapPartitions at PythonRDD.scala:168 []
 |  ShuffledRDD[3] at partitionBy at NativeMethodAccessorImpl.java:0 []
 +-(2) PairwiseRDD[2] at reduceByKey at /tmp/ipykernel_698/1771453912.py:2 []
    |  PythonRDD[1] at reduceByKey at /tmp/ipykernel_698/1771453912.py:2 []
    |  ParallelCollectionRDD[0] at readRDDFromFile at PythonRDD.scala:297 []
